# Civitas — train Model A (detector)

## WHAT YOU DO
1. On your PC, build the data (repo root):
   `python build_dataset.py`, then `python prepare_splits.py`, then `python scripts/package_data.py`.
   That creates `artifacts/civitas_data_v2.zip`.
2. Upload `civitas_data_v2.zip` to **Google Drive → My Drive → civitas/** (create the folder).
3. Get the code into Colab one of two ways:
   - push the repo to GitHub and paste its URL into cell 3, **or**
   - run `make package-code` on your PC and upload `artifacts/civitas_code.zip` to the
     same Drive folder.
4. **Runtime → Change runtime type → T4 GPU** (L4/A100 if you have Pro), then **Runtime → Run all**.
5. If Colab disconnects: reconnect and **Run all** again. Training resumes from the last save
   on Drive (at most ~500 steps are repeated).
6. When it finishes, paste back to Claude: the output of the **"Results to paste back"** cell,
   plus `data/DATA_REPORT.md` from your PC.
7. Do **not** run the last section ("RUN ONCE") until Claude says tuning is finished.

Rough time on a free T4: 40–60 min per epoch for ~300k rows; the default 4 epochs with early
stopping usually ends in 2.5–4 hours, across one or two sessions.

In [ ]:
# 1. GPU check: picks batch size / precision defaults for this GPU
import subprocess, torch
print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
                     capture_output=True, text=True).stdout)
assert torch.cuda.is_available(), "No GPU. Runtime > Change runtime type > T4 GPU, then run all again."
GPU = torch.cuda.get_device_name(0)
major, _ = torch.cuda.get_device_capability(0)
if "A100" in GPU:
    BATCH, ACCUM = 64, 1
elif "L4" in GPU:
    BATCH, ACCUM = 32, 1
else:  # T4 and anything else with ~15 GB
    BATCH, ACCUM = 32, 1
PRECISION = "bf16" if major >= 8 else "fp16 + GradScaler (T4 has no native bf16)"
print(f"{GPU}: batch {BATCH} x accum {ACCUM}, {PRECISION}")

In [ ]:
# 2. Google Drive: everything that must survive a disconnect lives here
from google.colab import drive
drive.mount("/content/drive")
PROJECT = "/content/drive/MyDrive/civitas"
import os
os.makedirs(PROJECT, exist_ok=True)
print("project folder:", PROJECT, "->", sorted(os.listdir(PROJECT)))

In [ ]:
# 3. Get the code: from GitHub (fill REPO_URL) OR from civitas_code.zip in the Drive folder
REPO_URL = ""  #@param {type:"string"}
BRANCH = "main"  #@param {type:"string"}
import os, subprocess, zipfile
CODE = "/content/civitas"
if REPO_URL:
    if os.path.exists(f"{CODE}/.git"):
        subprocess.run(["git", "-C", CODE, "pull", "--ff-only"], check=True)
    else:
        subprocess.run(["git", "clone", "--depth", "1", "-b", BRANCH, REPO_URL, CODE], check=True)
else:
    # made on your PC with `make package-code` (or any zip of the repo folder)
    zpath = f"{PROJECT}/civitas_code.zip"
    assert os.path.exists(zpath), f"Set REPO_URL above, or upload civitas_code.zip to {PROJECT}"
    zipfile.ZipFile(zpath).extractall(CODE)
    inner = [d for d in os.listdir(CODE) if os.path.exists(f"{CODE}/{d}/train.py")]
    if not os.path.exists(f"{CODE}/train.py") and inner:  # zip had a top-level folder
        CODE = f"{CODE}/{inner[0]}"
os.chdir(CODE)
print(subprocess.run(["git", "log", "--oneline", "-1"], capture_output=True, text=True).stdout
      or "code from zip")

In [ ]:
# 4. Install what training needs (Colab already has torch)
!pip -q install "transformers>=4.40" "indic-transliteration>=2.3.45" pyyaml pyarrow scikit-learn sentencepiece
import transformers, indic_transliteration
print("transformers", transformers.__version__)

In [ ]:
# 5. Unpack the data package and verify it against its manifest
import hashlib, json, zipfile, os
DATA = "/content/data_pkg"
zipfile.ZipFile(f"{PROJECT}/civitas_data_v2.zip").extractall(DATA)
manifest = json.load(open(f"{DATA}/manifest.json"))
for name, digest in manifest["files"].items():
    h = hashlib.sha256(open(f"{DATA}/{name}", "rb").read()).hexdigest()
    assert h == digest, f"{name} is corrupted (sha256 mismatch) - re-upload the zip"
for split, info in manifest["splits"].items():
    print(f"{split:>5}: {info['rows']:>7} rows  languages={info['language_counts']}")
print("built from commit", manifest["git_commit"], "at", manifest["created_at"])
# the packaged data_config.yaml carries the language sampling weights used for this data
if os.path.exists(f"{DATA}/data_config.yaml"):
    os.replace(f"{DATA}/data_config.yaml", f"{CODE}/data_config.yaml")

In [ ]:
# 6. Training config. Every key is a train.py flag (python train.py --help lists them all)
RUN_NAME = "model_a_v2"  #@param {type:"string"}
OUT = f"{PROJECT}/{RUN_NAME}"
CONFIG = {
    "data_dir": DATA,
    "output_dir": OUT,
    "encoder_name": "xlm-roberta-base",
    "epochs": 4,
    "batch_size": BATCH,
    "grad_accum": ACCUM,
    "lr": 2e-5,
    "max_length": 128,
    "warmup_ratio": 0.06,
    "evals_per_epoch": 3,
    "patience": 4,
    "save_every": 500,
    # loss
    "hate_gamma": 0.0,
    "severity_gamma": 2.0,
    "w_hate": 1.0, "w_target": 0.5, "w_severity": 0.5, "w_rationale": 0.5,
    # augmentation (train split only)
    "identity_aug": 1,
    "script_augment_p": 0.3,
    "spelling_augment_p": 0.15,
    "lang_weights": "data_config.yaml",
}
ARGS = " ".join(f"--{k} {v!r}" if isinstance(v, str) else f"--{k} {v}" for k, v in CONFIG.items())
print(ARGS)

In [ ]:
# 7. Train. Safe to re-run after a disconnect: --resume continues from OUT/last.pt
!cd {CODE} && python train.py {ARGS} --resume

## Results to paste back
Run the next cell and paste its whole output to Claude.

In [ ]:
# 8. Summary of the best checkpoint (val only) + behavioural probe on val
import json
m = json.load(open(f"{OUT}/best_metrics.json"))
keep = ["threshold", "hate_macro_f1", "hate_roc_auc", "severity_macro_f1", "target_macro_f1",
        "rationale_token_f1", "epoch", "global_step"]
print(json.dumps({k: m.get(k) for k in keep}, indent=1))
for s in ("hate_by_language", "hate_by_script", "hate_by_source"):
    print(s, {k: (round(v["macro_f1"], 3) if v else None) for k, v in m[s].items()})
print("target per class", {k: round(v["f1"], 2) for k, v in m["target_per_class"].items()})
!cd {CODE} && python -c "import pandas as pd; pd.read_parquet('{DATA}/val.parquet').to_csv('/content/val.csv', index=False)"
!cd {CODE} && python -m eval.probe_model --checkpoint {OUT}/best_model.pt --eval_csv /content/val.csv --out {OUT}/robustness.json

---
# RUN ONCE — final report
**Only run this after Claude confirms tuning is finished.** The test split is the one honest
number you get. If you change anything (data, config, code) after looking at these results,
the test numbers stop being trustworthy.

In [ ]:
# 9. Test split + HateCheck functional tests. RUN ONCE.
CONFIRM = False  #@param {type:"boolean"}
assert CONFIRM, "Tick CONFIRM only when tuning is finished (see the note above)."
!cd {CODE} && python train.py --eval_only --split test --data_dir {DATA} --output_dir {OUT} --batch_size {BATCH}
!cd {CODE} && python -m eval.eval_hatecheck --checkpoint {OUT}/best_model.pt --out {OUT}/hatecheck.json
print(open(f"{OUT}/test_report.json").read()[:3000])